<a href="https://colab.research.google.com/github/dowes48/A2E_polars/blob/main/A2E_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast A2E Using Polars - Google Colab Session

When you run one of the following Colab code cells for the first time, Google will warn you that this is not a Google notebook. Choose the option "Run Anyway".

For example, **Click on the arrow in the next cell** to determine what version of Python Google is using, then choose the option "Run Anyway".  

*one can also run a code cell by placing the cursor in the cell and keying "Shift-Enter"*

In [51]:
!python --version

Python 3.12.12


Python 3.12.12 was released in 10/2025 as a security fix for version 3.12 which was first released in 10/2023. Since the current stable version of Python is 3.14, one may occasionally find that the latest Python expressions may not work in Colab. See https://docs.python.org/release/3.12.12/ for version specific documentation.

*Colab can also be configured to run R code.*

---

I am using GitHub to store this Notebook and its associated data files. If you got this far, you already have a Colab copy of my notebook. Now you need to do next is to **click the arrow in the next cell** to make a temporary copy of my entire GitHub repository including the data files that we will want to use.

In [52]:
!git clone "https://github.com/dowes48/A2E_polars"

fatal: destination path 'A2E_polars' already exists and is not an empty directory.


After the cloning has finished, click on the File folder icon to the left. A folder for A2E_polars should appear. Under this, you will find my data folder.

---

This is a "Text" cell - actually created as Markdown language. Colab is very nice to use for **Markdown** because while the Markdown special characters are entered in the left-hand side of this cell, the right-hand side shows how the text will appear to the reader. Double-click on this cell to enter editing mode and you will see what I mean.

---

The following following cell *imports* the libraries that will be used in this notebook. The cells that follow depend on the libraries' code, so make certain this is run first. In general, run cells in the given sequence.

In [53]:
import os                                # basic file handling functions
import polars as pl                      # polars, a DataFrame library for manipulating structured data
from datetime import timedelta           # a single datatype needed for calculations
import time

In [54]:
# read the data file into a polars dataframe ('pl' is shorthand for polars)
df = pl.read_csv(r'A2E_polars/data/HRA_data.csv', try_parse_dates = True)

In [55]:
# rename exam year column
df = df.rename({'examY': 'year'})

# define Enumeration datatypes
enum_vs   = pl.Enum(["alive", "dead"])
enum_sex  = pl.Enum(["M", "F"])
enum_dx   = pl.Enum(["healthy", "cancer", "cardiac", "diabetes", "neuro", "pulm", "renal", "other"])
enum_site = pl.Enum(["AL", "AZ", "CA", "FL", "GA", "LA", "SC", "TX", "VA"])
enum_agebnd  = pl.Enum(["65 - 69", "70 - 74", "75 - 79", "80 - 84"])

# cast column values to defined datatypes
df = df.with_columns(
    pl.col("vs").cast(enum_vs),
    pl.col("sex").cast(enum_sex),
    pl.col("dxGrp").cast(enum_dx),
    pl.col("site").cast(enum_site),
    pl.col("ageBand").cast(enum_agebnd))

df = df.with_columns(
    pl.col("age").cast(pl.UInt8),
    pl.col("year").cast(pl.UInt16),
    pl.col("UID").cast(pl.UInt32),
    pl.col("mr").cast(pl.Float32))

print(df)

shape: (24_528, 12)
┌───────┬────────────┬────────────┬───────┬───┬──────┬──────┬──────┬─────────┐
│ UID   ┆ examD      ┆ exitD      ┆ vs    ┆ … ┆ year ┆ site ┆ mr   ┆ ageBand │
│ ---   ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---  ┆ ---  ┆ ---  ┆ ---     │
│ u32   ┆ date       ┆ date       ┆ enum  ┆   ┆ u16  ┆ enum ┆ f32  ┆ enum    │
╞═══════╪════════════╪════════════╪═══════╪═══╪══════╪══════╪══════╪═════════╡
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 1998 ┆ AL   ┆ 3.5  ┆ 70 - 74 │
│ 10002 ┆ 1998-01-05 ┆ 2005-12-11 ┆ dead  ┆ … ┆ 1998 ┆ TX   ┆ 3.92 ┆ 75 - 79 │
│ 10003 ┆ 1998-01-06 ┆ 2001-03-20 ┆ dead  ┆ … ┆ 1998 ┆ AZ   ┆ 5.46 ┆ 70 - 74 │
│ 10004 ┆ 1998-06-10 ┆ 2005-09-23 ┆ dead  ┆ … ┆ 1998 ┆ SC   ┆ 4.41 ┆ 75 - 79 │
│ 10005 ┆ 1998-01-09 ┆ 2003-12-28 ┆ dead  ┆ … ┆ 1998 ┆ GA   ┆ 4.9  ┆ 80 - 84 │
│ …     ┆ …          ┆ …          ┆ …     ┆ … ┆ …    ┆ …    ┆ …    ┆ …       │
│ 93526 ┆ 2007-12-26 ┆ 2008-06-30 ┆ alive ┆ … ┆ 2007 ┆ VA   ┆ 1.47 ┆ 65 - 69 │
│ 93528 ┆ 2007-12-27 ┆ 2008-06-3

In [56]:
# split off columns of "details"
df_details = df.select('UID', 'sex', 'mr', 'dxGrp', 'site', 'ageBand')
print("\n\ndetails dataframe 'df_details' variables will be re-joined asfter expansion")
print(df_details)

# keep columns needed for expansion
df = df.select('UID', 'examD', 'exitD', 'vs', 'age', 'year')
print("\nStudy dataframe 'df' now contains only variables needed for expansion")
print(df)



details dataframe 'df_details' variables will be re-joined asfter expansion
shape: (24_528, 6)
┌───────┬──────┬──────┬──────────┬──────┬─────────┐
│ UID   ┆ sex  ┆ mr   ┆ dxGrp    ┆ site ┆ ageBand │
│ ---   ┆ ---  ┆ ---  ┆ ---      ┆ ---  ┆ ---     │
│ u32   ┆ enum ┆ f32  ┆ enum     ┆ enum ┆ enum    │
╞═══════╪══════╪══════╪══════════╪══════╪═════════╡
│ 10001 ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10002 ┆ F    ┆ 3.92 ┆ cardiac  ┆ TX   ┆ 75 - 79 │
│ 10003 ┆ M    ┆ 5.46 ┆ pulm     ┆ AZ   ┆ 70 - 74 │
│ 10004 ┆ M    ┆ 4.41 ┆ cardiac  ┆ SC   ┆ 75 - 79 │
│ 10005 ┆ M    ┆ 4.9  ┆ other    ┆ GA   ┆ 80 - 84 │
│ …     ┆ …    ┆ …    ┆ …        ┆ …    ┆ …       │
│ 93526 ┆ F    ┆ 1.47 ┆ healthy  ┆ VA   ┆ 65 - 69 │
│ 93528 ┆ F    ┆ 0.98 ┆ healthy  ┆ TX   ┆ 75 - 79 │
│ 93529 ┆ M    ┆ 1.47 ┆ healthy  ┆ TX   ┆ 70 - 74 │
│ 93530 ┆ M    ┆ 0.98 ┆ healthy  ┆ AL   ┆ 70 - 74 │
│ 93531 ┆ M    ┆ 1.47 ┆ healthy  ┆ LA   ┆ 80 - 84 │
└───────┴──────┴──────┴──────────┴──────┴─────────┘

Study dataframe 'd

In [57]:
# tDelta constants to be used in exposure calculation
tDelta_1yr = timedelta(days=365, hours=6)
tDelta_1day  = timedelta(days=1)

# calculate total number of intervals for each subject
df = df.with_columns(totalIntvls = ((pl.col('exitD') - pl.col('examD') + tDelta_1day)/tDelta_1yr)
                     .ceil().cast(pl.Int32))
print(df)

shape: (24_528, 7)
┌───────┬────────────┬────────────┬───────┬─────┬──────┬─────────────┐
│ UID   ┆ examD      ┆ exitD      ┆ vs    ┆ age ┆ year ┆ totalIntvls │
│ ---   ┆ ---        ┆ ---        ┆ ---   ┆ --- ┆ ---  ┆ ---         │
│ u32   ┆ date       ┆ date       ┆ enum  ┆ u8  ┆ u16  ┆ i32         │
╞═══════╪════════════╪════════════╪═══════╪═════╪══════╪═════════════╡
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           │
│ 10002 ┆ 1998-01-05 ┆ 2005-12-11 ┆ dead  ┆ 79  ┆ 1998 ┆ 8           │
│ 10003 ┆ 1998-01-06 ┆ 2001-03-20 ┆ dead  ┆ 72  ┆ 1998 ┆ 4           │
│ 10004 ┆ 1998-06-10 ┆ 2005-09-23 ┆ dead  ┆ 75  ┆ 1998 ┆ 8           │
│ 10005 ┆ 1998-01-09 ┆ 2003-12-28 ┆ dead  ┆ 83  ┆ 1998 ┆ 6           │
│ …     ┆ …          ┆ …          ┆ …     ┆ …   ┆ …    ┆ …           │
│ 93526 ┆ 2007-12-26 ┆ 2008-06-30 ┆ alive ┆ 69  ┆ 2007 ┆ 1           │
│ 93528 ┆ 2007-12-27 ┆ 2008-06-30 ┆ alive ┆ 79  ┆ 2007 ┆ 1           │
│ 93529 ┆ 2007-12-28 ┆ 2008-06-30 ┆ alive ┆ 73  ┆ 2007 ┆ 1

In [58]:
# create temporary column of identical interval index lists
df = df.with_columns(idx_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])

# take slice of idx_list and store list in new column ‘intvl’
df = df.with_columns(intvl = pl.col('idx_list').list.slice(0, pl.col('totalIntvls')))
df = df.drop('idx_list')

#  explode to get desired tabular output
df = df.explode(pl.col('intvl'))

print(df)

shape: (92_727, 8)
┌───────┬────────────┬────────────┬───────┬─────┬──────┬─────────────┬───────┐
│ UID   ┆ examD      ┆ exitD      ┆ vs    ┆ age ┆ year ┆ totalIntvls ┆ intvl │
│ ---   ┆ ---        ┆ ---        ┆ ---   ┆ --- ┆ ---  ┆ ---         ┆ ---   │
│ u32   ┆ date       ┆ date       ┆ enum  ┆ u8  ┆ u16  ┆ i32         ┆ i64   │
╞═══════╪════════════╪════════════╪═══════╪═════╪══════╪═════════════╪═══════╡
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           ┆ 1     │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           ┆ 2     │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           ┆ 3     │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           ┆ 4     │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ 71  ┆ 1998 ┆ 7           ┆ 5     │
│ …     ┆ …          ┆ …          ┆ …     ┆ …   ┆ …    ┆ …           ┆ …     │
│ 93526 ┆ 2007-12-26 ┆ 2008-06-30 ┆ alive ┆ 69  ┆ 2007 ┆ 1           ┆ 1     │
│ 93528 ┆ 2007-12-27 ┆ 2008-06-30

In [59]:
# create temporary column of offset years, then calculate ‘intvl_date’
df = df.with_columns(offsetYrs = (pl.col("intvl") - 1).cast(pl.String))
df = df.with_columns(intvl_date = pl.col('examD').dt.offset_by((pl.col('offsetYrs')) + "y"),
        intvl_age = pl.col('age') + pl.col('intvl') - 1)
## df = df.filter(pl.col('exitD') > pl.col('intvl_date'))
print(df)


shape: (92_727, 11)
┌───────┬────────────┬────────────┬───────┬───┬───────┬───────────┬────────────┬───────────┐
│ UID   ┆ examD      ┆ exitD      ┆ vs    ┆ … ┆ intvl ┆ offsetYrs ┆ intvl_date ┆ intvl_age │
│ ---   ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---   ┆ ---       ┆ ---        ┆ ---       │
│ u32   ┆ date       ┆ date       ┆ enum  ┆   ┆ i64   ┆ str       ┆ date       ┆ i64       │
╞═══════╪════════════╪════════════╪═══════╪═══╪═══════╪═══════════╪════════════╪═══════════╡
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 1     ┆ 0         ┆ 1998-01-02 ┆ 71        │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 2     ┆ 1         ┆ 1999-01-02 ┆ 72        │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 3     ┆ 2         ┆ 2000-01-02 ┆ 73        │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 4     ┆ 3         ┆ 2001-01-02 ┆ 74        │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 5     ┆ 4         ┆ 2002-01-02 ┆ 75        │
│ …     ┆ …          ┆ …          ┆ …     ┆ … ┆ … 

In [60]:
# three important calculations: interval year, exposure, and actual
df = df.with_columns(
    # first the interval year (for acquiring qx)
    year = pl.col('intvl_date').dt.year(),

    # second the exposure
    persYrs = pl
        .when((pl.col('intvl_date').dt.offset_by('1y')) <= pl.col('exitD'))
        .then(pl.lit(1))
        .otherwise((pl.col('exitD') - pl.col('intvl_date') + tDelta_1day) / tDelta_1yr),

    # actual deaths
    actual = pl
        .when((pl.col('vs') == "dead") & (pl.col('intvl')
            == pl.col('totalIntvls')))
        .then(pl.lit(1))
        .otherwise(pl.lit(0)))
print(df)

shape: (92_727, 13)
┌───────┬────────────┬────────────┬───────┬───┬────────────┬───────────┬──────────┬────────┐
│ UID   ┆ examD      ┆ exitD      ┆ vs    ┆ … ┆ intvl_date ┆ intvl_age ┆ persYrs  ┆ actual │
│ ---   ┆ ---        ┆ ---        ┆ ---   ┆   ┆ ---        ┆ ---       ┆ ---      ┆ ---    │
│ u32   ┆ date       ┆ date       ┆ enum  ┆   ┆ date       ┆ i64       ┆ f64      ┆ i32    │
╞═══════╪════════════╪════════════╪═══════╪═══╪════════════╪═══════════╪══════════╪════════╡
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 1998-01-02 ┆ 71        ┆ 1.0      ┆ 0      │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 1999-01-02 ┆ 72        ┆ 1.0      ┆ 0      │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 2000-01-02 ┆ 73        ┆ 1.0      ┆ 0      │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 2001-01-02 ┆ 74        ┆ 1.0      ┆ 0      │
│ 10001 ┆ 1998-01-02 ┆ 2004-04-28 ┆ dead  ┆ … ┆ 2002-01-02 ┆ 75        ┆ 1.0      ┆ 0      │
│ …     ┆ …          ┆ …          ┆ …     ┆ … ┆ … 

In [61]:
# clean up and get ready for join
df = df.select('UID', 'vs', 'intvl', 'intvl_date', 'intvl_age',
                         'year', 'persYrs', 'actual')
df = df.rename({'intvl_age': 'age'})
df = df.join(df_details, on='UID', how='left')
print(df)

# load mortality dataframe
df_mort = pl.read_csv(r"A2E_polars/data/USA_both_1933-2023.psv",separator='|')
df_mort = df_mort.rename({'Year': 'year', 'Age': 'age'})
enum_sex  = pl.Enum(["M", "F"])
df_mort = df_mort.with_columns(pl.col('sex').cast(enum_sex))

print("\nHMD: USA, both sexes, 1933-2023")
print(df_mort)

# join to get qx
df = df.join(df_mort, on=('year','age','sex'), how='left')
print(df)


shape: (92_727, 13)
┌───────┬───────┬───────┬────────────┬───┬──────┬──────────┬──────┬─────────┐
│ UID   ┆ vs    ┆ intvl ┆ intvl_date ┆ … ┆ mr   ┆ dxGrp    ┆ site ┆ ageBand │
│ ---   ┆ ---   ┆ ---   ┆ ---        ┆   ┆ ---  ┆ ---      ┆ ---  ┆ ---     │
│ u32   ┆ enum  ┆ i64   ┆ date       ┆   ┆ f32  ┆ enum     ┆ enum ┆ enum    │
╞═══════╪═══════╪═══════╪════════════╪═══╪══════╪══════════╪══════╪═════════╡
│ 10001 ┆ dead  ┆ 1     ┆ 1998-01-02 ┆ … ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ dead  ┆ 2     ┆ 1999-01-02 ┆ … ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ dead  ┆ 3     ┆ 2000-01-02 ┆ … ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ dead  ┆ 4     ┆ 2001-01-02 ┆ … ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ dead  ┆ 5     ┆ 2002-01-02 ┆ … ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ …     ┆ …     ┆ …     ┆ …          ┆ … ┆ …    ┆ …        ┆ …    ┆ …       │
│ 93526 ┆ alive ┆ 1     ┆ 2007-12-26 ┆ … ┆ 1.47 ┆ healthy  ┆ VA   ┆ 65 - 69 │
│ 93528 ┆ alive ┆ 1     ┆ 2007-12-27 ┆ … ┆ 0

In [62]:
# last calculation, expected deaths
df = df.with_columns(expected = pl.col('persYrs') * pl.col('qx') * pl.col('mr')).sort('UID', 'intvl')

df = df.select('UID', 'age', 'sex', 'mr', 'dxGrp', 'site', 'ageBand', 'intvl', 'intvl_date', 'year',
               'persYrs', 'actual', 'qx', 'expected').sort('UID', 'intvl')

# -----------------------------------------------------
# print("\nthe 7 leftmost columns")
print(df[:,:7])

with pl.Config(
    tbl_cell_numeric_alignment="RIGHT",
    float_precision=5,
):
    print("\nthe 7 rightmost columns")
    print(df[:,7:])
# ----------------------------------------------------

df.write_csv(r'A2E_polars/data/a2e_results_' + '.csv')
# double-clicking the csv file will open it in MS Excel


shape: (92_727, 7)
┌───────┬─────┬──────┬──────┬──────────┬──────┬─────────┐
│ UID   ┆ age ┆ sex  ┆ mr   ┆ dxGrp    ┆ site ┆ ageBand │
│ ---   ┆ --- ┆ ---  ┆ ---  ┆ ---      ┆ ---  ┆ ---     │
│ u32   ┆ i64 ┆ enum ┆ f32  ┆ enum     ┆ enum ┆ enum    │
╞═══════╪═════╪══════╪══════╪══════════╪══════╪═════════╡
│ 10001 ┆ 71  ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ 72  ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ 73  ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ 74  ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ 10001 ┆ 75  ┆ M    ┆ 3.5  ┆ diabetes ┆ AL   ┆ 70 - 74 │
│ …     ┆ …   ┆ …    ┆ …    ┆ …        ┆ …    ┆ …       │
│ 93526 ┆ 69  ┆ F    ┆ 1.47 ┆ healthy  ┆ VA   ┆ 65 - 69 │
│ 93528 ┆ 79  ┆ F    ┆ 0.98 ┆ healthy  ┆ TX   ┆ 75 - 79 │
│ 93529 ┆ 73  ┆ M    ┆ 1.47 ┆ healthy  ┆ TX   ┆ 70 - 74 │
│ 93530 ┆ 72  ┆ M    ┆ 0.98 ┆ healthy  ┆ AL   ┆ 70 - 74 │
│ 93531 ┆ 84  ┆ M    ┆ 1.47 ┆ healthy  ┆ LA   ┆ 80 - 84 │
└───────┴─────┴──────┴──────┴──────────┴──────┴──────